# Experiment: Rule Application Oracle Demo


This notebook demonstrates one RuleArena airline sample end-to-end:

1. Load the original sample `info`.
2. Compute applicable coarse and fine rules.
3. Run the rule-application oracle trace (execution order).


In [1]:
from pathlib import Path
import importlib.util
import types
import sys
import json
from pprint import pprint


def load_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


# Resolve project root whether notebook is launched from repo root or notebooks/
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

print("Project root:", ROOT)

# Load segmentation and applicability modules directly (no heavy model imports)
seg_mod = load_module("src.rulearena.rulebook_segments", ROOT / "src/rulearena/rulebook_segments.py")
app_mod = load_module("src.rulearena.rule_applicability", ROOT / "src/rulearena/rule_applicability.py")

# Register lightweight package stubs so oracle's absolute imports resolve cleanly
src_pkg = types.ModuleType("src")
src_pkg.__path__ = [str(ROOT / "src")]
sys.modules["src"] = src_pkg

rulearena_pkg = types.ModuleType("src.rulearena")
rulearena_pkg.__path__ = [str(ROOT / "src/rulearena")]
sys.modules["src.rulearena"] = rulearena_pkg

sys.modules["src.rulearena.rulebook_segments"] = seg_mod
sys.modules["src.rulearena.rule_applicability"] = app_mod

oracle_mod = load_module("src.rulearena.rule_application_oracle", ROOT / "src/rulearena/rule_application_oracle.py")


Project root: /Users/vitoriag/Documents/multi-rules


In [2]:
# Choose one sample
COMP = 0
SAMPLE_IDX = 1

rulebook_path = ROOT / "datasets/RuleArena/airline/reference_rules.txt"
problems_path = ROOT / f"datasets/RuleArena/airline/synthesized_problems/comp_{COMP}.jsonl"

with open(problems_path) as f:
    for i, line in enumerate(f):
        if i == SAMPLE_IDX:
            sample = json.loads(line)
            break
    else:
        raise IndexError(f"Sample {SAMPLE_IDX} not found in {problems_path}")

info = sample["info"]
print(f"Loaded comp_{COMP} sample {SAMPLE_IDX}")


Loaded comp_0 sample 1


## Original Sample Info


In [3]:
pprint(info)


{'bag_list': [{'id': 1, 'name': 'backpack', 'size': [18, 13, 6], 'weight': 8},
              {'id': 2,
               'name': 'luggage box',
               'size': [41, 20, 16],
               'weight': 95},
              {'id': 3, 'name': 'backpack', 'size': [38, 24, 18], 'weight': 74},
              {'id': 4, 'name': 'backpack', 'size': [37, 16, 10], 'weight': 54},
              {'id': 5,
               'name': 'backpack',
               'size': [43, 25, 20],
               'weight': 52}],
 'base_price': 186,
 'customer_class': 'Business',
 'direction': 1,
 'routine': 'U.S.'}


## Applicable Coarse and Fine Rules


In [5]:
rulebook_text = rulebook_path.read_text()
coarse_segments = seg_mod.get_coarse_segments(rulebook_text)
fine_segments = seg_mod.get_fine_segments(rulebook_text)

applied = app_mod.get_applied_rules_with_coarse(info, fine_segments, coarse_segments)

print("Applicable coarse sections:", len(applied["coarse"]))
for i, seg in enumerate(applied["coarse"], start=1):
    print(f"{i:2d}. {seg['name']}")

print("Applicable fine rules:", len(applied["fine"]))
for i, seg in enumerate(applied["fine"], start=1):
    print(f"{i:2d}. {seg['name']}")


Applicable coarse sections: 9
 1. preamble
 2. carry_on
 3. checked_bags_intro
 4. first_bag
 5. second_bag
 6. third_bag
 7. fourth_bag
 8. complimentary_bags
 9. weight_and_size
Applicable fine rules: 29
 1. preamble/all_published_bag_fees
 2. carry_on/you_re_allowed_1
 3. carry_on/your_personal_item_like
 4. carry_on/these_don_t_count
 5. carry_on/you_can_bring_only
 6. carry_on/the_total_size_of
 7. carry_on/your_soft_sided_garment
 8. checked_bags_intro/bag_fees_have_been
 9. checked_bags_intro/travel_within_between_the
10. checked_bags_intro/travel_to_from_canada
11. checked_bags_intro/all_bag_fees_are
12. first_bag/row_us_puerto_rico
13. second_bag/row_us_canada_puerto
14. third_bag/row_us_canada_puerto
15. fourth_bag/row_us_canada_puerto
16. complimentary_bags/in_some_cases_you
17. complimentary_bags/if_your_status_level
18. complimentary_bags/free_checked_bags_may
19. complimentary_bags/1st_checked_bag_is
20. complimentary_bags/or_when_traveling_to
21. complimentary_bags/1st_a

## Oracle Output (Execution Order)


In [6]:
trace = oracle_mod.get_rule_application_trace(info, fine_segments)

print("Bag processing order (after complementary-gain reordering):")
for row in trace["bag_processing_order"]:
    print(
        f"  rank={row['rank']} original_checked_bag={row['original_bag_index']} "
        f"gain={row['complementary_gain']} size={row['total_size']} weight={row['weight']}"
    )

print("Rule application steps:")
for step in trace["steps"]:
    prefix = f"[{step['step_index']:02d}] {step['phase']}"
    bag_part = ""
    if step["phase"] == "bag":
        bag_part = f" bag_rank={step['bag_rank']} orig_bag={step['original_bag_index']}"
    fee_part = ""
    if step["computed_fee"] is not None:
        fee_part = f" fee={step['computed_fee']}"
    winner_part = f" winner={step['winner']}" if step["winner"] else ""

    print(f"{prefix}{bag_part}: {step['rule_name']}{fee_part}{winner_part}")


Bag processing order (after complementary-gain reordering):
  rank=1 original_checked_bag=3 gain=70 size=63 weight=54
  rank=2 original_checked_bag=1 gain=0 size=77 weight=95
  rank=3 original_checked_bag=2 gain=0 size=80 weight=74
  rank=4 original_checked_bag=4 gain=0 size=88 weight=52
Rule application steps:
[01] global: preamble/all_published_bag_fees
[02] global: carry_on/you_re_allowed_1
[03] global: carry_on/your_personal_item_like
[04] global: carry_on/these_don_t_count
[05] global: carry_on/you_can_bring_only
[06] global: carry_on/the_total_size_of
[07] global: carry_on/your_soft_sided_garment
[08] global: checked_bags_intro/bag_fees_have_been
[09] global: checked_bags_intro/travel_within_between_the
[10] global: checked_bags_intro/travel_to_from_canada
[11] global: checked_bags_intro/all_bag_fees_are
[12] global: complimentary_bags/in_some_cases_you
[13] global: complimentary_bags/if_your_status_level
[14] global: complimentary_bags/free_checked_bags_may
[15] global: complime

## Structured Oracle Payload (JSON)


In [7]:
print(json.dumps(trace, indent=2))


{
  "info": {
    "base_price": 186,
    "customer_class": "Business",
    "routine": "U.S.",
    "direction": 1,
    "bag_list": [
      {
        "id": 1,
        "name": "backpack",
        "size": [
          18,
          13,
          6
        ],
        "weight": 8
      },
      {
        "id": 2,
        "name": "luggage box",
        "size": [
          41,
          20,
          16
        ],
        "weight": 95
      },
      {
        "id": 3,
        "name": "backpack",
        "size": [
          38,
          24,
          18
        ],
        "weight": 74
      },
      {
        "id": 4,
        "name": "backpack",
        "size": [
          37,
          16,
          10
        ],
        "weight": 54
      },
      {
        "id": 5,
        "name": "backpack",
        "size": [
          43,
          25,
          20
        ],
        "weight": 52
      }
    ]
  },
  "bag_processing_order": [
    {
      "rank": 1,
      "original_bag_index": 3,
      "com